In [ ]:
# ============================================================
# BASELINE EXPERIMENTS FOR IRRJ PAPER
# Linear Regression + MLP + Neural ODE
# Auto-detect characters from Drive folder
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import json
import os
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr
from google.colab import drive

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/Research_Paper_2/Character_Details/Character_role_tag_with_embeddings"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# Dataset Class
# ------------------------------------------------------------
class EmbeddingDataset(Dataset):

    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


# ------------------------------------------------------------
# Load JSON and create layer transition pairs
# ------------------------------------------------------------
def load_character_data(file_path):

    with open(file_path) as f:
        data = json.load(f)

    X = []
    Y = []

    for item in data:

        p = item["parva"]
        ch = item["chapter"]
        sn = item["sentence_number"]

        embeddings = item["embeddings"]

        layers = sorted(
            embeddings.keys(),
            key=lambda x: int(x.split("_")[1])
        )

        for i in range(len(layers)-1):

            layer_id = int(layers[i].split("_")[1])

            emb_l = embeddings[layers[i]]
            emb_next = embeddings[layers[i+1]]

            metadata = [p, ch, sn, layer_id]

            X.append(metadata + emb_l)
            Y.append(emb_next)

    return np.array(X), np.array(Y)


# ------------------------------------------------------------
# Models
# ------------------------------------------------------------
class LinearModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(772,768)

    def forward(self,x):
        return self.linear(x)


class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(772,256),
            nn.GELU(),
            nn.Linear(256,256),
            nn.GELU(),
            nn.Linear(256,768)
        )

    def forward(self,x):
        return self.net(x)


class NeuralODE(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(772,256),
            nn.Tanh(),
            nn.Linear(256,256),
            nn.Tanh(),
            nn.Linear(256,768)
        )

    def forward(self,x):
        return self.net(x)


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------
def train_model(model, train_loader, epochs=100):

    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for X,Y in train_loader:

            X,Y = X.to(device),Y.to(device)

            optimizer.zero_grad()

            pred = model(X)

            loss = criterion(pred,Y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} Loss {total_loss/len(train_loader):.4f}")

    return model


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------
def evaluate_model(model,test_loader):

    model.eval()

    preds=[]
    trues=[]

    with torch.no_grad():

        for X,Y in test_loader:

            X=X.to(device)

            pred=model(X).cpu().numpy()
            Y=Y.numpy()

            preds.append(pred)
            trues.append(Y)

    preds=np.vstack(preds)
    trues=np.vstack(trues)

    mse=mean_squared_error(trues,preds)
    mae=mean_absolute_error(trues,preds)
    r2=r2_score(trues,preds)

    cos=np.mean([
        cosine_similarity([p],[t])[0][0]
        for p,t in zip(preds,trues)
    ])

    pear=pearsonr(preds.flatten(),trues.flatten())[0]

    return mse,mae,r2,cos,pear


# ------------------------------------------------------------
# Run Experiment For One Character
# ------------------------------------------------------------
def run_character_experiment(file_name):

    character = file_name.replace(".json","")

    print("\n====================================")
    print("Character:",character)

    file_path = os.path.join(DATA_PATH, file_name)

    X,Y = load_character_data(file_path)

    dataset = EmbeddingDataset(X,Y)

    torch.manual_seed(42)

    train_size = int(0.8*len(dataset))
    test_size = len(dataset)-train_size

    train_dataset,test_dataset = random_split(dataset,[train_size,test_size])

    train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
    test_loader = DataLoader(test_dataset,batch_size=32)

    models = {
        "Linear":LinearModel(),
        "MLP":MLP(),
        "NeuralODE":NeuralODE()
    }

    results = {}

    for name,model in models.items():

        print("\nTraining",name)

        model = train_model(model,train_loader)

        mse,mae,r2,cos,pear = evaluate_model(model,test_loader)

        results[name] = [mse,mae,r2,cos,pear]

    print("\nRESULT TABLE")
    print("Model | MSE | MAE | R2 | CosSim | Pearson")

    for k,v in results.items():

        print(k,"|",
              round(v[0],4),"|",
              round(v[1],4),"|",
              round(v[2],4),"|",
              round(v[3],4),"|",
              round(v[4],4))

    return results


# ------------------------------------------------------------
# Automatically detect all character files
# ------------------------------------------------------------
files = [f for f in os.listdir(DATA_PATH) if f.endswith(".json")]

print("Detected files:", files)

all_results = {}

for file in files:

    results = run_character_experiment(file)

    all_results[file.replace(".json","")] = results


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Detected files: ['Arjuna_role_tagged.json', 'Abhimanyu_role_tagged.json', 'Duryodhana_role_tagged.json', 'Karna_role_tagged.json', 'Krishna_role_tagged.json', 'Yudhishthira_role_tagged.json']

Character: Arjuna_role_tagged

Training Linear
Epoch 1 Loss 1817.9952
Epoch 2 Loss 8.1559
Epoch 3 Loss 3.9610
Epoch 4 Loss 1.4753
Epoch 5 Loss 0.4159
Epoch 6 Loss 0.1199
Epoch 7 Loss 0.0655
Epoch 8 Loss 0.0535
Epoch 9 Loss 0.0462
Epoch 10 Loss 0.0401
Epoch 11 Loss 0.0351
Epoch 12 Loss 0.0309
Epoch 13 Loss 0.0277
Epoch 14 Loss 0.0250
Epoch 15 Loss 0.0232
Epoch 16 Loss 0.0217
Epoch 17 Loss 0.0205
Epoch 18 Loss 0.0196
Epoch 19 Loss 0.0188
Epoch 20 Loss 0.0180
Epoch 21 Loss 0.0175
Epoch 22 Loss 0.0167
Epoch 23 Loss 0.0167
Epoch 24 Loss 0.0162
Epoch 25 Loss 0.0161
Epoch 26 Loss 0.0156
Epoch 27 Loss 0.0154
Epoch 28 Loss 0.0149
Epoch 29 Loss 0.0148
Epoch 30 Loss 0.0149
Epoch 3